# Nevada — Title 57 (Insurance) → `data/nevada/ins_codes/*.md`

Nevada’s **insurance** provisions are **Title 57 — INSURANCE** of the **Nevada Revised Statutes (NRS)**. On **Justia**, the Title 57 browse page is **[`/codes/nevada/title-57/`](https://law.justia.com/codes/nevada/title-57/)**; individual sections live under **`/codes/nevada/chapter-<CH>/statute-<slug>/`** (Justia uses **`statute-`**, not `section-`).

**Cloudflare** often blocks plain **`httpx`**; this notebook uses **`curl_cffi`** with **`impersonate="chrome120"`** (same pattern as **`hawaii.ipynb`** / **`nebraska.ipynb`**).

**Discovery:** Parse the Title 57 index HTML between **`TITLE 57`** and **`TITLE 58`** to collect only **insurance** chapter index URLs (`chapter-679a` … `chapter-697`). For each chapter page, collect every **`statute-…`** link under **`/codes/nevada/`**.

**Download:** text from **`div.primary-content`**, with Justia boilerplate stripped. Files are **`NRS_sec_<label>.md`** where **`<label>`** is the statute slug without the `statute-` prefix, with hyphens changed to underscores (e.g. `679a-010` → `NRS_sec_679a_010.md`).

**Config:** **`MAX_STATUTES`** caps downloads (**0** = all). **`MAX_CHAPTER_PAGES`** caps chapter index fetches during discovery (**0** = no cap). **`REUSE_DISCOVERED_URLS`** skips discovery when **`_nevada_title57_statute_urls.txt`** exists.

Run with the **`ins_ipynb/`** directory as cwd (so `Path("data")` resolves to **`ins_ipynb/data/`**), same as other state notebooks.

Then run **`python -m app.ingest`** from the **project root**.


In [1]:
%pip install -q curl_cffi beautifulsoup4


You should consider upgrading via the '/Users/apps/Downloads/ZProjects/RAG/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import re
import time
from pathlib import Path
from urllib.parse import urljoin, urlparse

from bs4 import BeautifulSoup
from curl_cffi import requests as curl_requests

BASE = "https://law.justia.com"
TITLE_INDEX = f"{BASE}/codes/nevada/title-57/"

OUT_DIR = Path("data") / "nevada" / "ins_codes"
OUT_DIR.mkdir(parents=True, exist_ok=True)

CURL_IMPERSONATE = "chrome120"
REQUEST_DELAY_SEC = 0.12
TIMEOUT = 60.0

MAX_STATUTES = 0
MAX_CHAPTER_PAGES = 0

SKIP_EXISTING = True

DISCOVERED_LIST = OUT_DIR / "_nevada_title57_statute_urls.txt"
REUSE_DISCOVERED_URLS = True


In [3]:
def curl_get(url: str) -> str:
    time.sleep(REQUEST_DELAY_SEC)
    r = curl_requests.get(url, impersonate=CURL_IMPERSONATE, timeout=TIMEOUT)
    r.raise_for_status()
    return r.text


def path_key(u: str) -> str:
    return urlparse(u).path.rstrip("/")


def title57_insurance_chapter_urls(html: str) -> list[str]:
    """Chapter index URLs for Title 57 only (between TITLE 57 and TITLE 58 headings)."""
    u57 = html.upper().find("TITLE 57")
    u58 = html.upper().find("TITLE 58")
    if u57 == -1 or u58 == -1 or u58 <= u57:
        raise RuntimeError("Could not locate TITLE 57 / TITLE 58 region on index page")
    blob = html[u57:u58]
    soup = BeautifulSoup(blob, "html.parser")
    out: list[str] = []
    seen: set[str] = set()
    for a in soup.find_all("a", href=True):
        absu = urljoin(TITLE_INDEX, a["href"])
        pk = path_key(absu).lower()
        if not pk.startswith("/codes/nevada/chapter-"):
            continue
        if "/statute-" in pk:
            continue
        if pk in seen:
            continue
        seen.add(pk)
        out.append(absu if absu.endswith("/") else absu + "/")
    out.sort()
    return out


def discover_statute_urls() -> list[str]:
    html = curl_get(TITLE_INDEX)
    chapters = title57_insurance_chapter_urls(html)
    print(f"Title 57 chapter index pages: {len(chapters)}")

    statutes: set[str] = set()
    fetches = 0
    for ch_url in chapters:
        if MAX_CHAPTER_PAGES and fetches >= MAX_CHAPTER_PAGES:
            break
        html_ch = curl_get(ch_url)
        fetches += 1
        soup = BeautifulSoup(html_ch, "html.parser")
        for a in soup.find_all("a", href=True):
            absu = urljoin(ch_url, a["href"])
            pk = path_key(absu).lower()
            if "/statute-" not in pk:
                continue
            if not pk.startswith("/codes/nevada/chapter-"):
                continue
            statutes.add(absu if absu.endswith("/") else absu + "/")
    return sorted(statutes)


def statute_label_from_url(url: str) -> str:
    path = path_key(url).lower()
    if "/statute-" not in path:
        raise ValueError(f"not a statute URL: {url!r}")
    tail = path.rsplit("/statute-", 1)[1]
    return tail


def label_sort_key(label: str) -> tuple:
    out: list[tuple[int, int | str]] = []
    for part in label.split("-"):
        if part.isdigit():
            out.append((0, int(part)))
        else:
            out.append((1, part.lower()))
    return tuple(out)


def label_to_display_citation(label: str) -> str:
    """679a-010 -> 679A.010 ; 689-090 -> 689.090"""
    m = re.match(r"^(\d+)([a-z])?-(\d+)$", label, re.I)
    if m:
        num, suf, sec = m.group(1), (m.group(2) or "").upper(), m.group(3)
        return f"{num}{suf}.{sec}"
    return label


def label_to_filename(label: str) -> str:
    safe = label.replace("-", "_")
    return f"NRS_sec_{safe}.md"


def extract_primary_text(html: str) -> tuple[str, str]:
    soup = BeautifulSoup(html, "html.parser")
    title_el = soup.find("title")
    title_txt = title_el.get_text(strip=True) if title_el else ""
    pc = soup.select_one("div.primary-content")
    if pc:
        text = pc.get_text("\n", strip=True)
    else:
        main = soup.find("main") or soup.find("article")
        text = main.get_text("\n", strip=True) if main else soup.get_text("\n", strip=True)
    return title_txt, text


def strip_justia_boilerplate(text: str) -> str:
    drop_prefixes = (
        "Go to Previous Versions",
        "View All Versions",
        "Learn more",
        "This media-neutral citation",
    )
    lines = text.split("\n")
    out: list[str] = []
    skip_until_substantive = True
    for line in lines:
        s = line.strip()
        if not s:
            if not skip_until_substantive:
                out.append("")
            continue
        if any(s.startswith(p) for p in drop_prefixes):
            continue
        if s.startswith("20") and ("Nevada Rev" in s or "Nevada Revised" in s):
            continue
        if s in {"Next", "Previous", "Universal Citation:"}:
            continue
        skip_until_substantive = False
        out.append(s)
    return "\n".join(out).strip()


def download_title57() -> dict[str, int]:
    if REUSE_DISCOVERED_URLS and DISCOVERED_LIST.exists() and DISCOVERED_LIST.stat().st_size > 50:
        raw = [ln.strip() for ln in DISCOVERED_LIST.read_text(encoding="utf-8").splitlines() if ln.strip()]
        all_urls = sorted(raw, key=lambda u: label_sort_key(statute_label_from_url(u)))
        print(f"Loaded {len(all_urls)} statute URLs from {DISCOVERED_LIST.name} (skipped discovery)")
    else:
        found = discover_statute_urls()
        print(f"Discovered {len(found)} statute URLs under Title 57")
        all_urls = sorted(found, key=lambda u: label_sort_key(statute_label_from_url(u)))
        DISCOVERED_LIST.write_text("\n".join(all_urls) + "\n", encoding="utf-8")

    todo = all_urls if not MAX_STATUTES else all_urls[:MAX_STATUTES]
    if MAX_STATUTES:
        print(f"Limited downloads to first {len(todo)} statutes (MAX_STATUTES)")

    wrote = skipped = failed = 0
    for i, st_url in enumerate(todo, 1):
        label = statute_label_from_url(st_url)
        disp = label_to_display_citation(label)
        dest = OUT_DIR / label_to_filename(label)
        if SKIP_EXISTING and dest.exists() and dest.stat().st_size > 80:
            skipped += 1
        else:
            try:
                html = curl_get(st_url)
                head_t, body_t = extract_primary_text(html)
                body_t = strip_justia_boilerplate(body_t)
                title = head_t or f"Nevada Revised Statutes § {disp}"
                md = (
                    f"# {title}\n\n"
                    f"**Nevada Revised Statutes — Title 57 (Insurance)**\n\n"
                    f"**Source (Justia mirror):** {st_url}\n\n"
                    f"**Verify on official NRS:** [leg.state.nv.us — NRS](https://www.leg.state.nv.us/nrs/nrs.html)\n\n"
                    f"**Section (URL slug):** statute-{label}\n\n"
                    f"**Citation (display):** NRS {disp}\n\n"
                    f"---\n\n"
                    f"{body_t}\n"
                )
                dest.write_text(md, encoding="utf-8")
                wrote += 1
            except Exception as e:
                print(f"FAIL {label}: {e}")
                failed += 1
        if i % 200 == 0:
            print(f"… {i}/{len(todo)} (wrote={wrote} skipped={skipped} failed={failed})")

    print(f"Done. wrote={wrote} skipped={skipped} failed={failed} → {OUT_DIR.resolve()}")
    return {"wrote": wrote, "skipped": skipped, "failed": failed}


download_title57()


Title 57 chapter index pages: 61
Discovered 3271 statute URLs under Title 57
… 200/3271 (wrote=200 skipped=0 failed=0)
… 400/3271 (wrote=400 skipped=0 failed=0)
… 600/3271 (wrote=600 skipped=0 failed=0)
… 800/3271 (wrote=800 skipped=0 failed=0)
… 1000/3271 (wrote=1000 skipped=0 failed=0)
… 1200/3271 (wrote=1200 skipped=0 failed=0)
… 1400/3271 (wrote=1400 skipped=0 failed=0)
… 1600/3271 (wrote=1600 skipped=0 failed=0)
… 1800/3271 (wrote=1800 skipped=0 failed=0)
… 2000/3271 (wrote=2000 skipped=0 failed=0)
… 2200/3271 (wrote=2200 skipped=0 failed=0)
… 2400/3271 (wrote=2400 skipped=0 failed=0)
… 2600/3271 (wrote=2600 skipped=0 failed=0)
… 2800/3271 (wrote=2800 skipped=0 failed=0)
… 3000/3271 (wrote=3000 skipped=0 failed=0)
… 3200/3271 (wrote=3200 skipped=0 failed=0)
Done. wrote=3271 skipped=0 failed=0 → /Users/apps/Downloads/ZProjects/RAG/ins_ipynb/data/nevada/ins_codes


{'wrote': 3271, 'skipped': 0, 'failed': 0}

## Next step

`python -m app.ingest` from the project root.
